# 02 — Features: design matrix for the X-learner

**Project:** Training ROI Predictor (H08)

Three numeric columns and two categoricals: `pre_perf`, `tenure_yrs`, `role_level`, `dept`, `training_id`. The X-learner runs four base learners and a propensity head over the same preprocessor, so the design matrix has to remain stable across them.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from training_roi.data import PROCESSED, load_panel, make_training_artifacts
from training_roi.features import NUMERIC, CATEGORICAL, build_preprocessor, feature_cols

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

In [ ]:
PARQUET = PROCESSED / 'training_outcomes.parquet'
df = load_panel() if PARQUET.exists() else make_training_artifacts()
df.head(3)

## 1. Build the preprocessor

In [ ]:
cols = feature_cols(df)
pre = build_preprocessor()
X = pre.fit_transform(df[cols])
print('design matrix shape:', X.shape)

## 2. Numeric column distributions after StandardScaler

In [ ]:
numeric_X = pd.DataFrame(X[:, :len(NUMERIC)], columns=NUMERIC)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, col in zip(axes, NUMERIC):
    sns.histplot(numeric_X[col], bins=30, ax=ax, color='#3a7ca5')
    ax.set_title(col)
plt.tight_layout(); plt.show()
numeric_X.describe().round(2)

## 3. Categorical one-hot expansion

In [ ]:
ohe = pre.named_steps['pre'].named_transformers_['cat'].named_steps['ohe']
ohe_names = ohe.get_feature_names_out(CATEGORICAL).tolist()
print(f'one-hot columns: {len(ohe_names)}')
print('first 8 OHE names:', ohe_names[:8])

## 4. Treatment-vs-control covariate balance

Standardised mean differences (SMD) per covariate. SMD > 0.1 indicates non-trivial imbalance — exactly the structural ingredient that motivates X-learner over a t-learner.

In [ ]:
smd_rows = []
for col in NUMERIC:
    t = df.loc[df.treatment == 1, col]
    c = df.loc[df.treatment == 0, col]
    sd = np.sqrt((t.var() + c.var()) / 2)
    smd_rows.append({'covariate': col, 'smd': (t.mean() - c.mean()) / sd})
smd_df = pd.DataFrame(smd_rows)
fig, ax = plt.subplots(figsize=(7, 3.4))
sns.barplot(data=smd_df, x='covariate', y='smd', ax=ax, color='#9c6644')
ax.axhline(0.1, color='red', ls='--', label='SMD = 0.1 (rule of thumb)')
ax.set_title('Standardised mean difference (treated vs control)')
ax.legend()
plt.tight_layout(); plt.show()

## 5. Outcome variance vs noise floor

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
sns.histplot(df, x='perf_uplift', bins=40, ax=ax, color='#4a7c59')
ax.axvline(df['perf_uplift'].mean(), color='red', ls='--', label='mean')
ax.set_title('Observed perf_uplift distribution')
ax.legend()
plt.tight_layout(); plt.show()

## 6. Treatment × department × training-id interaction

In [ ]:
interact = df.groupby(['dept', 'training_id'])['true_cate'].mean().unstack()
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(interact, cmap='RdYlGn', center=0.5, annot=True, fmt='.2f', ax=ax)
ax.set_title('True CATE by dept × training_id')
plt.tight_layout(); plt.show()

## 7. Hand-off

The featurisation choices above feed `models.fit_x_learner` directly. The next notebook walks the X-learner stack and quantifies the MAE against ground-truth CATE.